# Telugu Movie Worldwide Gross Prediction — Machine Learning Model




## 1. Install and import libraries

In [6]:
!pip -q install pandas openpyxl scikit-learn joblib

import io, re, warnings
import numpy as np
import pandas as pd
from google.colab import files
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
print('Libraries ready.')

Libraries ready.


## 2. Upload your movie dataset
Upload `telugu_movie_prediction_dataset.xlsx` (or your corrected workbook). Do not type missing values from memory.

In [7]:
uploaded = files.upload()
if not uploaded:
    raise ValueError('Upload the Excel dataset first.')
filename = next(iter(uploaded))
xls = pd.ExcelFile(io.BytesIO(uploaded[filename]))
sheet = 'Movie_Dataset' if 'Movie_Dataset' in xls.sheet_names else xls.sheet_names[0]
df = pd.read_excel(io.BytesIO(uploaded[filename]), sheet_name=sheet)
df.columns = [str(c).strip() for c in df.columns]
display(df.head())
print('Rows:', len(df), '| Columns:', list(df.columns))

Saving telugu_movie_prediction_dataset.xlsx to telugu_movie_prediction_dataset (1).xlsx


,Movie,Hero,Director,Budget_₹_crore,Worldwide_Gross_₹_crore,Certification,Release_Status,Notes,Recovery_%,Source_Budget_URL,Source_Gross_URL
0,Devara: Part 1,Jr NTR,Koratala Siva,300.0,510.0,U/A,Released,From handwritten notes; verify sources.,170.000000,NaN,NaN
1,RRR,Jr NTR; Ram Charan,S. S. Rajamouli,550.0,1300.0,U/A,Released,Shared film; included once with both heroes.,236.363636,NaN,NaN
2,Aravinda Sametha Veera Raghava,Jr NTR,Trivikram Srinivas,90.0,165.0,U/A,Released,From handwritten notes; verify sources.,183.333333,NaN,NaN
3,Pushpa 2: The Rule,Allu Arjun,Sukumar,450.0,1742.0,U/A,Released,From handwritten notes; verify sources.,387.111111,NaN,NaN
4,Pushpa: The Rise,Allu Arjun,Sukumar,175.0,360.0,U/A,Released,From handwritten notes; verify sources.,205.714286,NaN,NaN


Rows: 38 | Columns: ['Movie', 'Hero', 'Director', 'Budget_₹_crore', 'Worldwide_Gross_₹_crore', 'Certification', 'Release_Status', 'Notes', 'Recovery_%', 'Source_Budget_URL', 'Source_Gross_URL']


## 3. Validate and prepare the data
The model learns to predict **Worldwide_Gross_₹_crore** from budget, hero, and director. Upcoming/unclear movies are not training examples because their target gross is unknown or unverified.

## 4. Train a real ML regression model
We use a Random Forest regressor. Hero and director are categorical inputs; budget is numeric. Multi-hero strings are split into individual hero names and multi-hot encoded. A held-out test set provides a basic check of generalization.

In [8]:
REQUIRED = ['Movie', 'Hero', 'Director', 'Budget_₹_crore', 'Worldwide_Gross_₹_crore']
missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}. Please use the expected workbook columns.')

# Treat every record as a released film. Release_Status is intentionally NOT used as a filter.
for c in ['Budget_₹_crore', 'Worldwide_Gross_₹_crore']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

valid = (
    df['Budget_₹_crore'].notna() & (df['Budget_₹_crore'] > 0) &
    df['Worldwide_Gross_₹_crore'].notna() & (df['Worldwide_Gross_₹_crore'] >= 0) &
    df['Hero'].notna() & df['Hero'].astype(str).str.strip().ne('') &
    df['Director'].notna() & df['Director'].astype(str).str.strip().ne('')
)
train_df = df.loc[valid].copy()
excluded_df = df.loc[~valid].copy()

print(f'Usable supervised training rows: {len(train_df)} / {len(df)}')
print(f'Rows flagged because required training values are missing/invalid: {len(excluded_df)}')
print('Release_Status was not used to exclude any row.')
display(excluded_df[['Movie','Hero','Director','Budget_₹_crore','Worldwide_Gross_₹_crore','Release_Status','Notes']
                    if 'Notes' in df.columns else
                    ['Movie','Hero','Director','Budget_₹_crore','Worldwide_Gross_₹_crore','Release_Status']])
if len(train_df) < 8:
    raise ValueError('Too few usable records for a basic train/test evaluation. Add verified numeric gross values; do not invent missing data.')


Usable supervised training rows: 34 / 38
Rows flagged because required training values are missing/invalid: 4
Release_Status was not used to exclude any row.


,Movie,Hero,Director,Budget_₹_crore,Worldwide_Gross_₹_crore,Release_Status,Notes
25,War 2,Jr NTR,Ayan Mukerji,NaN,NaN,Needs verification,Handwriting unclear; verify title/values.
26,Peddi,Ram Charan,Buchi Babu Sana,350.0,NaN,Upcoming / verify,Gross blank; do not use as historical record.
27,The Paradise,Nani,Srikanth Odela,180.0,NaN,Upcoming / verify,Gross blank; do not use as historical record.
33,Ustaad Bhagat Singh,Pawan Kalyan,Harish Shankar,110.0,NaN,Upcoming / verify,Gross blank; do not use as historical record.


In [9]:
def split_names(value):
    if pd.isna(value): return []
    return [x.strip() for x in re.split(r';|,|\\band\\b', str(value), flags=re.I) if x.strip()]

# Normalize common NTR naming variants while retaining the original source table.
def normalize_hero_name(x):
    s = str(x).strip()
    aliases = {'ntr':'Jr NTR', 'jr ntr':'Jr NTR', 'junior ntr':'Jr NTR'}
    return aliases.get(s.lower(), s)

train_df['Hero_list'] = train_df['Hero'].apply(lambda x: [normalize_hero_name(h) for h in split_names(x)])
train_df['Director_clean'] = train_df['Director'].astype(str).str.strip()
mlb = MultiLabelBinarizer()
hero_matrix = mlb.fit_transform(train_df['Hero_list'])
hero_features = pd.DataFrame(hero_matrix, columns=[f'hero__{h}' for h in mlb.classes_], index=train_df.index)
X = pd.concat([train_df[['Budget_₹_crore']].rename(columns={'Budget_₹_crore':'budget_crore'}), hero_features,
               pd.get_dummies(train_df['Director_clean'], prefix='director', dtype=int)], axis=1)
y = train_df['Worldwide_Gross_₹_crore'].astype(float)

# With very small datasets, keep a test set but interpret scores cautiously.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = RandomForestRegressor(n_estimators=400, min_samples_leaf=2, max_features=0.9, random_state=42)
model.fit(X_train, y_train)
test_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, test_pred)
rmse = mean_squared_error(y_test, test_pred) ** 0.5
r2 = r2_score(y_test, test_pred) if len(y_test) >= 2 else float('nan')
metrics = pd.DataFrame([{'Metric':'MAE (₹ crore; lower is better)','Value':mae},
                        {'Metric':'RMSE (₹ crore; lower is better)','Value':rmse},
                        {'Metric':'R² (higher can be better; unstable on tiny test sets)','Value':r2},
                        {'Metric':'Training rows','Value':len(X_train)}, {'Metric':'Test rows','Value':len(X_test)}])
display(metrics)
display(pd.DataFrame({'Actual gross (₹ crore)':y_test, 'Predicted gross (₹ crore)':np.round(test_pred,1),
                      'Absolute error (₹ crore)':np.round(np.abs(y_test.to_numpy()-test_pred),1)}, index=y_test.index))
print('This is trained ML: model.fit(X_train, y_train) learned parameters from historical examples.')

,Metric,Value
0,MAE (₹ crore; lower is better),91.017431
1,RMSE (₹ crore; lower is better),99.136823
2,R² (higher can be better; unstable on tiny tes...,0.757297
3,Training rows,25.000000
4,Test rows,9.000000


,Actual gross (₹ crore),Predicted gross (₹ crore),Absolute error (₹ crore)
15,700.0,654.5,45.5
19,110.0,176.0,66.0
30,5.0,131.6,126.6
29,250.0,131.6,118.4
8,180.0,146.9,33.1
24,10.0,131.6,121.6
21,250.0,93.3,156.7
12,35.0,117.9,82.9
36,200.0,131.6,68.4


This is trained ML: model.fit(X_train, y_train) learned parameters from historical examples.


## 5. Ask the agent for a new movie
Enter the hero, director, and upcoming budget. The trained model predicts worldwide gross directly. If a hero/director was never seen during training, the notebook flags that limitation; unknown names cannot magically provide a learned effect.

In [11]:
hero_input = input('Enter hero name (for multiple heroes, separate with semicolons): ').strip()
director_input = input('Enter director name: ').strip()
budget_input = input('Enter upcoming movie budget in ₹ crore: ').strip()
try:
    budget_input = float(budget_input)
except ValueError:
    raise ValueError('Budget must be a number in ₹ crore.')
if not hero_input or not director_input or budget_input <= 0:
    raise ValueError('Enter a hero, director, and a positive budget.')

def build_input_row(hero, director, budget):
    heroes = [normalize_hero_name(h) for h in split_names(hero)]
    row = {c:0 for c in X.columns}
    row['budget_crore'] = budget
    for h in heroes:
        col = f'hero__{h}'
        if col in row: row[col] = 1
    dcol = f'director_{director.strip()}'
    if dcol in row: row[dcol] = 1
    return pd.DataFrame([row], columns=X.columns), heroes

new_X, entered_heroes = build_input_row(hero_input, director_input, budget_input)
predicted_gross = max(0.0, float(model.predict(new_X)[0]))
known_heroes = set(mlb.classes_)
unknown_heroes = [h for h in entered_heroes if h not in known_heroes]
known_directors = set(train_df['Director_clean'])
warnings_list = []
if unknown_heroes: warnings_list.append('Hero(s) not seen in training: ' + ', '.join(unknown_heroes))
if director_input.strip() not in known_directors: warnings_list.append('Director not seen in training: ' + director_input.strip())
display(Markdown(f'## Prediction- **Hero:** {hero_input}- **Director:** {director_input}- **Budget:** ₹{budget_input:,.2f} crore- **ML predicted worldwide gross:** **₹{predicted_gross:,.2f} crore**'))
if warnings_list:
    display(Markdown('**Data limitations**\\n' + '\\n'.join('- ' + w for w in warnings_list)))


Enter hero name (for multiple heroes, separate with semicolons): Mahesh Babu
Enter director name: S. S. Rajamouli
Enter upcoming movie budget in ₹ crore: 700


## Prediction- **Hero:** Mahesh Babu- **Director:** S. S. Rajamouli- **Budget:** ₹700.00 crore- **ML predicted worldwide gross:** **₹1,211.05 crore**